In [1]:
%load_ext sql

In [2]:
%%sql
sqlite:////content/travel.sqlite

# All table information -

In [3]:
%%sql
SELECT *
FROM sqlite_master
WHERE type='table';

 * sqlite:////content/travel.sqlite
Done.


type,name,tbl_name,rootpage,sql
table,aircrafts_data,aircrafts_data,2,"CREATE TABLE aircrafts_data ( aircraft_code character(3) NOT NULL, model jsonb NOT NULL, range integer NOT NULL, CONSTRAINT aircrafts_range_check CHECK ((range > 0)) )"
table,airports_data,airports_data,3,"CREATE TABLE airports_data ( airport_code character(3) NOT NULL, airport_name jsonb NOT NULL, city jsonb NOT NULL, coordinates point NOT NULL, timezone text NOT NULL )"
table,boarding_passes,boarding_passes,4,"CREATE TABLE boarding_passes ( ticket_no character(13) NOT NULL, flight_id integer NOT NULL, boarding_no integer NOT NULL, seat_no character varying(4) NOT NULL )"
table,bookings,bookings,5,"CREATE TABLE bookings ( book_ref character(6) NOT NULL, book_date timestamp with time zone NOT NULL, total_amount numeric(10,2) NOT NULL )"
table,flights,flights,6,"CREATE TABLE flights ( flight_id integer NOT NULL, flight_no character(6) NOT NULL, scheduled_departure timestamp with time zone NOT NULL, scheduled_arrival timestamp with time zone NOT NULL, departure_airport character(3) NOT NULL, arrival_airport character(3) NOT NULL, status character varying(20) NOT NULL, aircraft_code character(3) NOT NULL, actual_departure timestamp with time zone, actual_arrival timestamp with time zone )"
table,seats,seats,7,"CREATE TABLE seats ( aircraft_code character(3) NOT NULL, seat_no character varying(4) NOT NULL, fare_conditions character varying(10) NOT NULL )"
table,ticket_flights,ticket_flights,8,"CREATE TABLE ticket_flights ( ticket_no character(13) NOT NULL, flight_id integer NOT NULL, fare_conditions character varying(10) NOT NULL, amount numeric(10,2) NOT NULL )"
table,tickets,tickets,9,"CREATE TABLE tickets ( ticket_no character(13) NOT NULL, book_ref character(6) NOT NULL, passenger_id character varying(20) NOT NULL)"


# When the airport codes are put together, they form the route or flight path. Grouping by both the departure and arrival cities, we can see which combinations have the most amount of instances.

By joining 3 tables - flights, airports_data (aliased as departure) and airports_data (aliased as arrival), we can retrieve flight details along with origin and destination airport cities and coordinates.

In [4]:
%%sql
SELECT flights.flight_id, departure.airport_code as departure_airport_code, departure.airport_name as departure_airport, departure.city as departure_city,
arrival.airport_code as arrival_airport_code, arrival.airport_name as arrival_airport, arrival.city as arrival_city
from flights
left join airports_data as departure
on flights.departure_airport = departure.airport_code
left join airports_data as arrival
on flights.arrival_airport = arrival.airport_code
limit 5;

 * sqlite:////content/travel.sqlite
Done.


flight_id,departure_airport_code,departure_airport,departure_city,arrival_airport_code,arrival_airport,arrival_city
1185,DME,"{""en"": ""Domodedovo International Airport"", ""ru"": ""Домодедово""}","{""en"": ""Moscow"", ""ru"": ""Москва""}",BTK,"{""en"": ""Bratsk Airport"", ""ru"": ""Братск""}","{""en"": ""Bratsk"", ""ru"": ""Братск""}"
3979,VKO,"{""en"": ""Vnukovo International Airport"", ""ru"": ""Внуково""}","{""en"": ""Moscow"", ""ru"": ""Москва""}",HMA,"{""en"": ""Khanty Mansiysk Airport"", ""ru"": ""Ханты-Мансийск""}","{""en"": ""Khanty-Mansiysk"", ""ru"": ""Ханты-Мансийск""}"
4739,VKO,"{""en"": ""Vnukovo International Airport"", ""ru"": ""Внуково""}","{""en"": ""Moscow"", ""ru"": ""Москва""}",AER,"{""en"": ""Sochi International Airport"", ""ru"": ""Сочи""}","{""en"": ""Sochi"", ""ru"": ""Сочи""}"
5502,SVO,"{""en"": ""Sheremetyevo International Airport"", ""ru"": ""Шереметьево""}","{""en"": ""Moscow"", ""ru"": ""Москва""}",UFA,"{""en"": ""Ufa International Airport"", ""ru"": ""Уфа""}","{""en"": ""Ufa"", ""ru"": ""Уфа""}"
6938,SVO,"{""en"": ""Sheremetyevo International Airport"", ""ru"": ""Шереметьево""}","{""en"": ""Moscow"", ""ru"": ""Москва""}",ULV,"{""en"": ""Ulyanovsk Baratayevka Airport"", ""ru"": ""Баратаевка""}","{""en"": ""Ulyanovsk"", ""ru"": ""Ульяновск""}"


As you can see, the data inside those rows is in JSONs and needs to be extracted to used. I am going to store the table with the extracted information in a temporary table to use throughout the code.

In [5]:
%%sql
CREATE TEMPORARY TABLE airports_data_updated AS
SELECT * FROM airports_data;

 * sqlite:////content/travel.sqlite
Done.


[]

In [6]:
%%sql
UPDATE airports_data_updated
SET
  airport_name = JSON_EXTRACT(airport_name, '$.en'),
  city = JSON_EXTRACT(city, '$.en');

 * sqlite:////content/travel.sqlite
104 rows affected.


[]

Re-running the previous code using the airports_data_updated table.

In [7]:
%%sql
SELECT flights.flight_id, departure.airport_code as departure_airport_code, departure.airport_name as departure_airport, departure.city as departure_city,
arrival.airport_code as arrival_airport_code, arrival.airport_name as arrival_airport, arrival.city as arrival_city
from flights
left join airports_data_updated as departure
on flights.departure_airport = departure.airport_code
left join airports_data_updated as arrival
on flights.arrival_airport = arrival.airport_code
limit 5;

 * sqlite:////content/travel.sqlite
Done.


flight_id,departure_airport_code,departure_airport,departure_city,arrival_airport_code,arrival_airport,arrival_city
1185,DME,Domodedovo International Airport,Moscow,BTK,Bratsk Airport,Bratsk
3979,VKO,Vnukovo International Airport,Moscow,HMA,Khanty Mansiysk Airport,Khanty-Mansiysk
4739,VKO,Vnukovo International Airport,Moscow,AER,Sochi International Airport,Sochi
5502,SVO,Sheremetyevo International Airport,Moscow,UFA,Ufa International Airport,Ufa
6938,SVO,Sheremetyevo International Airport,Moscow,ULV,Ulyanovsk Baratayevka Airport,Ulyanovsk


## Let's see how the class of flight affects the price.

In [8]:
%%sql
SELECT fare_conditions AS 'fare_class', COUNT(*) as tickets_sold, ROUND(AVG(amount),1) as avg_ticket_cost, ROUND(SUM(amount),1) as revenue_earned
from ticket_flights
group by fare_conditions;

 * sqlite:////content/travel.sqlite
Done.


fare_class,tickets_sold,avg_ticket_cost,revenue_earned
Business,107642,51143.4,5505179600.0
Comfort,17291,32740.6,566116900.0
Economy,920793,15959.8,14695684400.0


Economy class makes the most amount of revenue despite the cost per ticket being the lowest due to the sheer number of tickets bought.

# Which airports bring in the most revenue?

In [9]:
%%sql
SELECT f.departure_airport AS airport_code, a.airport_name, SUM(tf.amount) AS revenue
FROM flights f
JOIN ticket_flights tf ON f.flight_id = tf.flight_id
JOIN tickets t ON tf.ticket_no = t.ticket_no
JOIN bookings b ON t.book_ref = b.book_ref
JOIN airports_data_updated a ON f.departure_airport = a.airport_code
GROUP BY f.departure_airport
ORDER BY revenue DESC;

 * sqlite:////content/travel.sqlite
Done.


airport_code,airport_name,revenue
DME,Domodedovo International Airport,3489450200
SVO,Sheremetyevo International Airport,3005036300
LED,Pulkovo Airport,1469442200
KHV,Khabarovsk-Novy Airport,1309975700
OVB,Tolmachevo Airport,1274457100
VKO,Vnukovo International Airport,968456800
IKT,Irkutsk Airport,663253700
AER,Sochi International Airport,546738000
SVX,Koltsovo Airport,514530000
PEE,Bolshoye Savino Airport,433534400
